# Dynamic ODLS on OCMR cardiac cine -- Colab runner

Runs `odls_dynamic/`: ODLS (arXiv:2112.04721) extended to dynamic MRI, with the
paper's soft-threshold placement (between Psi2 and Psi3) -- see `odls_dynamic/README.md`.

**Run top to bottom. Every step is safe to re-run after a disconnect:**
- preprocessing skips files it already finished,
- training resumes from the exact batch where it stopped,
- everything persistent lives in `DRIVE_ROOT` on your Drive (~5 GB data + ~0.1 GB per run).

Raw OCMR data never touches your laptop or Drive: each raw file is streamed from
OCMR's public S3 bucket onto this Colab VM's temporary disk, processed, and deleted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Config -- the only cell you normally edit

In [ ]:
# Everything persistent goes under here (Google Drive).
DRIVE_ROOT = "/content/drive/MyDrive/odls_dynamic"
DRIVE_DATA_DIR = f"{DRIVE_ROOT}/ocmr_preprocessed"   # written once, ~4.8 GB (hard cap below)
MAX_DATA_GB = 8                                      # preprocessing stops before exceeding this

# Temporary, wiped when the runtime resets -- that's fine.
LOCAL_DATA_DIR = "/content/ocmr_preprocessed"        # fast local copy used for training
RAW_TMP_DIR = "/content/ocmr_tmp"                    # holds one raw OCMR file at a time

REPO_URL = "https://github.com/Shambhawi419/1D_MRI.git"
REPO_DIR = "/content/1D_MRI"
CODE_DIR = f"{REPO_DIR}/odls_dynamic"

# Experiment. A new RUN_NAME = a new, separate run folder; the same RUN_NAME = resume.
AF = 4                     # acceleration (4, then 8 is the usual cine benchmark pair)
KERNEL_T = 1               # 1 = frame-by-frame baseline (current experiment); 3 = dynamic model
SPARSE_DOMAIN = "yt"       # "yt" (paper) or "yf" (temporal-Fourier sparsity, k-t style)
RUN_NAME = f"dyn_af{AF:g}_kt{KERNEL_T}_{SPARSE_DOMAIN}_v2"
CHECKPOINT_DIR = f"{DRIVE_ROOT}/runs/{RUN_NAME}"

EPOCHS = 40                # the dynamic AF 4 run plateaued around epoch 20-30
BATCH_SIZE = 32            # rows of a median-size slice; scaled per slice to keep GPU memory flat
TIME_LIMIT_MIN = 180       # stop cleanly + flush Drive every 3 h; re-run Train to continue
EXTRA_TRAIN_ARGS = ""      # e.g. "--init-checkpoint /content/drive/MyDrive/odls_dynamic/runs/<run>/best.pt"

In [ ]:
import os, subprocess, sys
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
sys.path.insert(0, CODE_DIR)

from google.colab import drive


def flush_drive():
    # Blocks until everything written to Drive has finished uploading. Colab uploads
    # in the background; files still queued are LOST if the runtime is killed (e.g. when
    # the free GPU quota runs out), so Train and Evaluate call this when they end.
    print("flushing Google Drive (waiting for uploads to finish) ...")
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    print("Drive flushed -- everything is saved.")


!nvidia-smi --query-gpu=name,memory.total --format=csv

## Optional: self-test (1-2 min, no data needed)
Run after pulling new code, before spending GPU time.

In [ ]:
!cd {CODE_DIR} && python -u selftest.py

## 1. Preprocess OCMR (one time, ~30-60 min)
Streams the 128 fully-sampled, symmetric-echo, single-slice SAX + LAX series from
OCMR's S3 bucket one at a time and writes ~4.8 GB to `DRIVE_DATA_DIR`. If the
runtime disconnects, just run this cell again -- finished files are skipped.

The train/val/test split (102/13/13 files) is fixed in `split.csv` on the first run.
For the paper-style mismatch experiment (test on unseen 0.55T scans), run once with
`--split-mode holdout-scanner --resplit-only` -- no re-download needed.

In [ ]:
!cd {CODE_DIR} && python -u preprocess_ocmr.py --out-dir {DRIVE_DATA_DIR} --tmp-dir {RAW_TMP_DIR} --max-gb {MAX_DATA_GB}

## 2. Copy the preprocessed data to local disk (every new session, ~1-2 min)
Training reads many small row slices; local disk is much faster than Drive for that.

If Google Drive's connection drops mid-copy ("Transport endpoint is not connected"),
this cell reconnects Drive and carries on by itself; files already copied are skipped.
Check that it ends with **128 slices available locally** before training (fewer means
preprocessing hasn't finished -- re-run step 1).

In [ ]:
import shutil, time
from google.colab import drive


def remount():
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    drive.mount('/content/drive', force_remount=True)


src_dir, dst_dir = f"{DRIVE_DATA_DIR}/slices", f"{LOCAL_DATA_DIR}/slices"
os.makedirs(dst_dir, exist_ok=True)
for attempt in range(1, 6):
    try:
        names = sorted(n for n in os.listdir(src_dir) if n.endswith(".npy"))
        copied = 0
        for name in names:
            src, dst = f"{src_dir}/{name}", f"{dst_dir}/{name}"
            if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
                continue                          # already copied
            shutil.copy(src, dst + ".tmp")        # copy to a temp name, then rename,
            os.replace(dst + ".tmp", dst)         # so a drop never leaves a half-copied file
            copied += 1
        for name in ("manifest.csv", "split.csv"):
            shutil.copy(f"{DRIVE_DATA_DIR}/{name}", f"{LOCAL_DATA_DIR}/{name}")
        print(f"done: copied {copied} new files, {len(names)} slices available locally")
        break
    except OSError as e:
        print(f"Drive connection dropped ({e}); reconnecting, attempt {attempt}/5 ...")
        time.sleep(5)
        remount()
else:
    raise RuntimeError("Drive kept disconnecting -- restart the runtime and run this cell again")

## 3. Train
Re-running this cell with the same `RUN_NAME` resumes exactly where it stopped
(model, optimizer, scheduler, AMP scaler, RNG, epoch and batch position).
Checkpoints: every 200 steps and every epoch, to `CHECKPOINT_DIR` on Drive.
`history.csv` there has per-epoch losses and every phase's eta1 / eta2 / lambda.

**Drive safety:** Colab uploads Drive files in the background. If the runtime is killed
before an upload finishes (e.g. the free GPU quota runs out), that file is lost. So this
cell stops training cleanly after `TIME_LIMIT_MIN` and then **flushes Drive** (waits until
every checkpoint is uploaded) -- also if you stop the cell yourself. Re-run it to continue.

In [ ]:
train_cmd = (f"cd {CODE_DIR} && python -u train.py --data-dir {LOCAL_DATA_DIR} "
             f"--checkpoint-dir {CHECKPOINT_DIR} --af {AF} --kernel-t {KERNEL_T} "
             f"--sparse-domain {SPARSE_DOMAIN} --epochs {EPOCHS} --batch-size {BATCH_SIZE} "
             f"--time-limit-min {TIME_LIMIT_MIN} --num-workers 2 {EXTRA_TRAIN_ARGS}")
try:
    get_ipython().system(train_cmd)
finally:
    flush_drive()   # runs however training ends: finished, time limit, error, or you pressed stop

## 4. Evaluate on the held-out test split
The model is rebuilt from `best.pt` alone (no architecture flags). Prints the model
and the zero-filled baseline side by side; writes
`CHECKPOINT_DIR/eval_test_af{AF}/results.json` plus a few example reconstructions.

In [ ]:
!cd {CODE_DIR} && python -u evaluate.py --data-dir {LOCAL_DATA_DIR} --checkpoint {CHECKPOINT_DIR}/best.pt --split test
flush_drive()

## 5. Look at a reconstruction

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
ex = sorted(glob.glob(f"{CHECKPOINT_DIR}/eval_test_af{AF:g}/*.npz"))[0]
d = np.load(ex)
ref, zf, rec = (d[k].astype(float) for k in ("ref", "zero_filled", "odls_dynamic"))
t, col = ref.shape[0] // 2, ref.shape[2] // 2
vmax = ref.max()
fig, ax = plt.subplots(2, 4, figsize=(16, 8))
panels = [(ref, "fully sampled"), (zf, f"zero-filled AF={AF}"), (rec, "dynamic ODLS"), (np.abs(rec - ref) * 5, "|error| x5")]
for j, (img, title) in enumerate(panels):
    ax[0, j].imshow(img[t], cmap="gray", vmin=0, vmax=vmax); ax[0, j].set_title(f"{title}, frame {t}")
    ax[1, j].imshow(img[:, :, col].T, cmap="gray", vmin=0, vmax=vmax, aspect="auto"); ax[1, j].set_title("y-t profile")
for a in ax.ravel():
    a.axis("off")
plt.suptitle(os.path.basename(ex)); plt.tight_layout(); plt.show()

## 6. Compare runs
Collects every `results.json` under `DRIVE_ROOT/runs`. The key comparison is
dynamic (`KERNEL_T=3`) vs frame-by-frame (`KERNEL_T=1`, same pipeline) vs zero-filled:
the gap between the first two is what using the time dimension buys.

To create the static baseline: set `KERNEL_T = 1` in the Config cell, re-run it, then
re-run cells 3 and 4 (it gets its own `RUN_NAME` folder automatically).

In [ ]:
import json
for path in sorted(glob.glob(f"{DRIVE_ROOT}/runs/*/eval_*/results.json")):
    r = json.load(open(path))
    run = path.split("/runs/")[1].split("/")[0]
    for m, o in r["overall"].items():
        print(f"{run:28s} {r['split']:5s} AF{r['af_requested']:<4g} {m:13s} "
              f"RLNE {o['RLNE_mean'] * 100:6.2f}  PSNR {o['PSNR_mean']:6.2f}  SSIM {o['SSIM_mean'] * 100:6.2f}")

## Other workflows from saved checkpoints
- **Fine-tune at another AF** (e.g. AF 8 from the AF 4 model): set `AF = 8` in Config
  (new `RUN_NAME`), and set `EXTRA_TRAIN_ARGS = "--init-checkpoint /content/drive/MyDrive/odls_dynamic/runs/dyn_af4_kt3_yt_v2/best.pt"`.
- **Train longer**: raise `EPOCHS` and re-run cell 3 -- it continues the same run.
- **Use a model in your own code**:
  `from checkpoint_utils import build_model_from_checkpoint; model, ckpt = build_model_from_checkpoint(path, "cuda")`
- **Different split** (e.g. the 0.55T mismatch test):
  `!cd {CODE_DIR} && python -u preprocess_ocmr.py --out-dir {DRIVE_DATA_DIR} --split-mode holdout-scanner --resplit-only`,
  then re-run cell 2 and train with a new `RUN_NAME`.